<a href="https://colab.research.google.com/github/paridebaiunco05-code/microwakeword-trainer/blob/main/FILEFINALEverooo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Custom wake-word trainer — bulletproof Colab notebook (2026 edition)

Train your own openWakeWord model in ~75-90 minutes on Colab Pro. **Run all → walk away → download `.onnx`** at the end.

## Why this notebook exists

openWakeWord's official Colab notebook bit-rotted hard against Python 3.12 (no piper-phonemize wheels), against torchaudio 2.x (`set_audio_backend` removed), against newer Colab images (HF Hub timeouts, namespace-package edge cases), and against its own internal config schema (4 keys silently moved to required-or-`KeyError` over the past year).

This notebook is patched against all of that. Every cell is **self-healing** — checks its own outputs, re-creates what's missing. There's a hard pre-flight gate (cell 4) that catches every dep/file issue before any of the slow downloads start.

It also **replaces openwakeword's `auto_train` with a faithful hand-rolled PyTorch loop** that mirrors auto_train's curriculum exactly: 3-stage learning rate (1e-4 → 1e-5 → 1e-6), negative-weight ramp 1→1500, hard-negative mining, FP-per-hour validation against an ACAV100M continuous-audio slice, 90/90/10 percentile checkpoint ensemble averaging. The upstream `auto_train` path keeps surfacing bugs against `mmap_batch_generator` shape handling on current openwakeword `master`; the hand-rolled trainer in cell 14 is small enough to read in one sitting and debug if anything ever changes.

## Run order

1. **Runtime → Change runtime type → L4 GPU + High RAM** (Colab Pro, $10/mo). Free T4 also works but is ~2× slower.
2. **Cell 10 — edit `TARGET_PHRASE` and `MODEL_NAME`** for your wake word. (Default = `mr graves`, the example we trained with.)
3. **Runtime → Run all**
4. Walk away ~75-90 min. Last cell downloads `<MODEL_NAME>.onnx`.

## Validated example (Mr Graves, 2026-05-09)

13 real-world utterances on a Pixel — every utterance fired:

```
WAKE — score=0.99664426
WAKE — score=0.59243464
WAKE — score=0.65439160
WAKE — score=0.77693284
WAKE — score=0.85433600
WAKE — score=0.64701974
WAKE — score=0.80747485
WAKE — score=0.95243360
WAKE — score=0.99938180
WAKE — score=0.90104705
WAKE — score=0.99639570
WAKE — score=0.98879445
WAKE — score=0.51673140
```

13/13 fires, scores 0.52-0.999, **zero phantom fires** during 30 min of normal phone use (TV, music, conversation, kitchen noise). Matches openwakeword's pretrained Hey Jarvis baseline (0.984/0.989) on clear utterances.

For deeper context on what auto_train actually does and why every part of the curriculum matters, the comments in cell 14 walk through each piece.

## 1. Comprehensive install â€” apt + pip + verify

Every dep + transitive we've discovered. `piper-tts` goes LAST via `--no-deps` so no later install can clobber it (piper-tts pulls regular `piper-phonemize` which has no Py 3.12 wheel; `piper-phonemize-cross` provides the same module name and is installed first).

In [121]:
# Native deps
!apt-get install -y -qq cmake espeak-ng espeak-ng-data libespeak-ng-dev \
    libsndfile1 pkg-config build-essential ffmpeg unzip 2>&1 | tail -3

# Python deps
!pip install -q \
    webrtcvad \
    mutagen==1.47.0 \
    torchinfo \
    torchmetrics \
    pyyaml \
    tqdm \
    datasets \
    soundfile \
    audiomentations==0.33.0 \
    torch_audiomentations \
    pronouncing \
    onnxruntime \
    onnx \
    speechbrain \
    acoustics \
    scipy \
    requests \
    huggingface_hub

# Piper moderno: supporta direttamente le voci .onnx
!pip install -q "piper-tts==1.3.0"

# Verify imports
import sys
print("Python:", sys.version)

import torch, torchinfo, torchmetrics, scipy, numpy
print(f"  torch: {torch.__version__}  cuda: {torch.cuda.is_available()}")

from tqdm import tqdm
import yaml, mutagen, pronouncing
import torchaudio, audiomentations, torch_audiomentations
import speechbrain, acoustics
import onnx, onnxruntime, soundfile, requests

from piper import PiperVoice, SynthesisConfig
from piper.phonemize_espeak import EspeakPhonemizer

print("  Piper 1.3.0 + dipendenze importate correttamente.")
print("  ONNX providers:", onnxruntime.get_available_providers())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.8/76.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 260.1/260.1 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 87.8 MB/s eta 0:00:00
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
  torch: 2.11.0+cu130  cuda: True
  Piper 1.3.0 + dipendenze importate correttamente.
  ONNX providers: ['AzureExecutionProvider', 'CPUExecutionProvider']


## 2. Clone repos + download model (self-healing)

Each piece (piper-sample-generator clone, libritts .pt model, openwakeword clone) is checked individually and re-created if missing or partial. Pin piper-sample-generator to commit `1a8c49bd^` â€” the parent of "Move to package", which is the last revision with flat-layout `generate_samples.py` at the root that openwakeword's train.py expects to import.

In [122]:
import os
import sys
import json

os.chdir('/content')

# ============================================================
# Italian Piper voices
# ============================================================

VOICE_DIR = '/content/piper_italian_voices'
GEN_DIR   = '/content/piper_italian_generator'

os.makedirs(VOICE_DIR, exist_ok=True)
os.makedirs(GEN_DIR, exist_ok=True)

ITALIAN_VOICES = {
    'paola': {
        'onnx': f'{VOICE_DIR}/it_IT-paola-medium.onnx',
        'json': f'{VOICE_DIR}/it_IT-paola-medium.onnx.json',
        'url_onnx': 'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx?download=true',
        'url_json': 'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/paola/medium/it_IT-paola-medium.onnx.json?download=true',
    },
    'riccardo': {
        'onnx': f'{VOICE_DIR}/it_IT-riccardo-x_low.onnx',
        'json': f'{VOICE_DIR}/it_IT-riccardo-x_low.onnx.json',
        'url_onnx': 'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx?download=true',
        'url_json': 'https://huggingface.co/rhasspy/piper-voices/resolve/main/it/it_IT/riccardo/x_low/it_IT-riccardo-x_low.onnx.json?download=true',
    }
}

for name, v in ITALIAN_VOICES.items():

    if not os.path.exists(v['onnx']) or os.path.getsize(v['onnx']) < 5_000_000:
        print(f'  downloading {name} ONNX...')
        !wget -q --tries=5 --timeout=300 -O "{v['onnx']}" "{v['url_onnx']}"

    if not os.path.exists(v['json']) or os.path.getsize(v['json']) < 1000:
        print(f'  downloading {name} JSON...')
        !wget -q --tries=5 --timeout=300 -O "{v['json']}" "{v['url_json']}"

    assert os.path.exists(v['onnx']), f'Missing {name} ONNX'
    assert os.path.exists(v['json']), f'Missing {name} JSON'

    with open(v['json'], encoding='utf-8') as f:
        meta = json.load(f)

    print(
        f"  ✓ {name}: "
        f"language={meta.get('language', {}).get('code', '?')}  "
        f"sample_rate={meta['audio']['sample_rate']}  "
        f"quality={meta['audio']['quality']}"
    )

# ============================================================
# openWakeWord
# ============================================================

OWW_DIR   = '/content/openwakeword'
OWW_TRAIN = f'{OWW_DIR}/openwakeword/train.py'

if not os.path.exists(OWW_TRAIN):
    print('  openwakeword missing/broken — re-cloning')
    !rm -rf "{OWW_DIR}"
    !git clone -q https://github.com/dscripka/openwakeword "{OWW_DIR}"
    !pip install -q -e "{OWW_DIR}"

assert os.path.exists(OWW_TRAIN), 'openWakeWord train.py missing'

# Make package importable
if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)

for _m in list(sys.modules):
    if _m.startswith('openwakeword'):
        del sys.modules[_m]

import openwakeword

assert openwakeword.__file__ is not None, (
    f'openwakeword imported as namespace package: {openwakeword.__path__}'
)

# Make the Italian generator directory importable
if GEN_DIR not in sys.path:
    sys.path.insert(0, GEN_DIR)

print(f'  ✓ openwakeword: {openwakeword.__file__}')
print(f'  ✓ Italian voices stored in: {VOICE_DIR}')
print(f'  ✓ generator adapter directory: {GEN_DIR}')

  downloading paola ONNX...
  downloading paola JSON...
  ✓ paola: language=it_IT  sample_rate=22050  quality=medium
  downloading riccardo ONNX...
  downloading riccardo JSON...
  ✓ riccardo: language=it_IT  sample_rate=16000  quality=x_low
  ✓ openwakeword: /content/openwakeword/openwakeword/__init__.py
  ✓ Italian voices stored in: /content/piper_italian_voices
  ✓ generator adapter directory: /content/piper_italian_generator


## 3. Apply runtime patches (idempotent)

Six patches. Each is a string check + targeted modification â€” re-running this cell after a partial install reapplies cleanly without double-patching.

In [123]:
import os
import glob

GEN_DIR   = '/content/piper_italian_generator'
VOICE_DIR = '/content/piper_italian_voices'
TRAIN_PY  = '/content/openwakeword/openwakeword/train.py'

# ============================================================
# PATCH A
# torch_audiomentations -> torchaudio.set_audio_backend
# ============================================================

for path in glob.glob('/usr/local/lib/python*/dist-packages/torch_audiomentations/utils/io.py'):
    with open(path, 'r', encoding='utf-8') as f:
        content = f.read()

    old = 'torchaudio.set_audio_backend("soundfile")'
    if old in content:
        content = content.replace(old, 'pass  # patched for torchaudio 2.x')
        with open(path, 'w', encoding='utf-8') as f:
            f.write(content)

        print(f'  Patch A applied: {path}')

# ============================================================
# PATCH B
# Italian Piper adapter
#
# openWakeWord expects:
#     from generate_samples import generate_samples
#
# We provide that API using Piper .onnx Italian voices.
# ============================================================

WRAPPER_PATH = f'{GEN_DIR}/generate_samples.py'

wrapper_code = r'''
import itertools as it
import os
import wave
from pathlib import Path

import onnxruntime as ort
from piper import PiperVoice, SynthesisConfig


ITALIAN_VOICES = [
    "/content/piper_italian_voices/it_IT-paola-medium.onnx",
    "/content/piper_italian_voices/it_IT-riccardo-x_low.onnx",
]

_USE_CUDA = (
    "CUDAExecutionProvider" in ort.get_available_providers()
)

_LOADED_VOICES = {}


def _load_voice(path):
    if path not in _LOADED_VOICES:
        _LOADED_VOICES[path] = PiperVoice.load(
            path,
            use_cuda=_USE_CUDA
        )
    return _LOADED_VOICES[path]


def generate_samples(
    text,
    output_dir,
    model=None,
    max_samples=None,
    file_names=None,
    batch_size=1,
    slerp_weights=(0.5,),
    length_scales=(0.75, 1.0, 1.25),
    noise_scales=(0.667,),
    noise_scale_ws=(0.8,),
    **kwargs,
):
    """
    Compatibility adapter for openWakeWord train.py.

    It generates Italian speech using Piper ONNX voices.
    batch_size / slerp_weights / auto_reduce_batch_size are accepted
    for compatibility with openWakeWord but are not required by this
    ONNX path.
    """

    if max_samples is None:
        if isinstance(text, list):
            max_samples = len(text)
        else:
            max_samples = 1

    model_paths = model or ITALIAN_VOICES
    voices = [_load_voice(str(p)) for p in model_paths]

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    if isinstance(text, list):
        texts = it.cycle(text)
    else:
        texts = it.cycle([text])

    if file_names:
        names = it.cycle(file_names)
    else:
        names = None

    settings = it.cycle(
        it.product(
            voices,
            length_scales,
            noise_scales,
            noise_scale_ws,
        )
    )

    for sample_idx in range(max_samples):

        voice, length_scale, noise_scale, noise_scale_w = next(settings)

        if names is not None:
            filename = next(names)
        else:
            filename = f"{sample_idx}.wav"

        output_path = output_dir / filename

        with wave.open(str(output_path), "wb") as wav_file:

            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(voice.config.sample_rate)

            voice.synthesize_wav(
                next(texts),
                wav_file=wav_file,
                syn_config=SynthesisConfig(
                    speaker_id=0,
                    length_scale=length_scale,
                    noise_scale=noise_scale,
                    noise_w_scale=noise_scale_w,
                ),
            )
'''

with open(WRAPPER_PATH, 'w', encoding='utf-8') as f:
    f.write(wrapper_code)

print(f'  Patch B: Italian Piper adapter created -> {WRAPPER_PATH}')

# ============================================================
# PATCH C
# HuggingFace timeouts
# ============================================================

os.environ['HF_HUB_ETAG_TIMEOUT'] = '120'
os.environ['HF_HUB_DOWNLOAD_TIMEOUT'] = '120'

import huggingface_hub.constants as hfc

for attr in [
    'DEFAULT_ETAG_TIMEOUT',
    'DEFAULT_DOWNLOAD_TIMEOUT',
    'HF_HUB_ETAG_TIMEOUT',
    'HF_HUB_DOWNLOAD_TIMEOUT'
]:
    if hasattr(hfc, attr):
        setattr(hfc, attr, 120)

print('  Patch C: HF Hub timeouts -> 120s')

# ============================================================
# PATCH D
# torchaudio.info shim
# ============================================================

import torchaudio

init_path = torchaudio.__file__
SHIM_MARKER = '# PATCH: info() shim for torchaudio 2.x'

with open(init_path, encoding='utf-8') as f:
    content = f.read()

if SHIM_MARKER not in content:

    shim = (
        f'\n\n{SHIM_MARKER}\n'
        'def info(file_path, *args, **kwargs):\n'
        '    import soundfile as _sf\n'
        '    si = _sf.info(str(file_path))\n'
        '    return type("_TorchaudioInfo", (), {\n'
        '        "num_frames": si.frames,\n'
        '        "sample_rate": si.samplerate,\n'
        '        "num_channels": si.channels,\n'
        '        "bits_per_sample": 16,\n'
        '        "encoding": "PCM_S",\n'
        '    })()\n'
    )

    with open(init_path, 'a', encoding='utf-8') as f:
        f.write(shim)

    import importlib
    importlib.reload(torchaudio)

print(f'  Patch D: torchaudio.info available = {hasattr(torchaudio, "info")}')

# ============================================================
# PATCH E
# Disable English-only adversarial phrase generator.
#
# We use custom Italian negative phrases from the config instead.
# ============================================================

ITALIAN_NEG_MARKER = '# ITALIAN NEGATIVE PHRASE PATCH'

with open(TRAIN_PY, 'r', encoding='utf-8') as f:
    train_content = f.read()

if ITALIAN_NEG_MARKER not in train_content:

    old_import = (
        'from openwakeword.data import generate_adversarial_texts, '
        'augment_clips, mmap_batch_generator'
    )

    new_import = (
        'from openwakeword.data import generate_adversarial_texts as _english_generate_adversarial_texts, '
        'augment_clips, mmap_batch_generator\n\n'
        f'{ITALIAN_NEG_MARKER}\n'
        'def generate_adversarial_texts(*args, **kwargs):\n'
        '    return []\n'
    )

    assert old_import in train_content, (
        'Expected openWakeWord import line not found; '
        'train.py changed upstream.'
    )

    train_content = train_content.replace(old_import, new_import, 1)

    with open(TRAIN_PY, 'w', encoding='utf-8') as f:
        f.write(train_content)

print('  Patch E: English-only adversarial generator disabled')

# ============================================================
# PATCH F
# train.py validation dtype cast
# ============================================================

with open(TRAIN_PY, 'r', encoding='utf-8') as f:
    train_content = f.read()

old_val = 'val_predictions = self.model(x_val)'
new_val = 'val_predictions = self.model(x_val.float())'

if old_val in train_content:
    train_content = train_content.replace(old_val, new_val)
    with open(TRAIN_PY, 'w', encoding='utf-8') as f:
        f.write(train_content)
    print('  Patch F: train.py validation dtype cast applied')
else:
    print('  Patch F: already applied or upstream code changed')

print()
print('All Italian compatibility patches applied.')

  Patch B: Italian Piper adapter created -> /content/piper_italian_generator/generate_samples.py
  Patch C: HF Hub timeouts -> 120s
  Patch D: torchaudio.info available = True
  Patch E: English-only adversarial generator disabled
  Patch F: train.py validation dtype cast applied

All Italian compatibility patches applied.


## 4. Pre-flight â€” every dep imports + every external file exists

Hard-fails if anything's wrong. This is the gate before slow downloads start. Catches new failure modes future Colab updates introduce.

In [124]:
import os
import importlib
import sys
import torch

VOICE_DIR = '/content/piper_italian_voices'
GEN_DIR   = '/content/piper_italian_generator'
OWW_DIR   = '/content/openwakeword'
TRAIN_PY  = '/content/openwakeword/openwakeword/train.py'

expected = {
    'Italian Paola ONNX':
        f'{VOICE_DIR}/it_IT-paola-medium.onnx',

    'Italian Paola JSON':
        f'{VOICE_DIR}/it_IT-paola-medium.onnx.json',

    'Italian Riccardo ONNX':
        f'{VOICE_DIR}/it_IT-riccardo-x_low.onnx',

    'Italian Riccardo JSON':
        f'{VOICE_DIR}/it_IT-riccardo-x_low.onnx.json',

    'Italian generate_samples adapter':
        f'{GEN_DIR}/generate_samples.py',

    'openwakeword train.py':
        TRAIN_PY,

    'openwakeword data.py':
        f'{OWW_DIR}/openwakeword/data.py',

    'openwakeword utils.py':
        f'{OWW_DIR}/openwakeword/utils.py',
}

missing_files = []

for name, p in expected.items():

    if os.path.exists(p):

        size = os.path.getsize(p)

        if size > 1e6:
            print(f'  ✓ {name}: {size/1e6:.1f} MB')
        else:
            print(f'  ✓ {name}: {size/1e3:.1f} KB')

    else:
        missing_files.append((name, p))
        print(f'  ✗ MISSING: {name} ({p})')

# ============================================================
# Imports
# ============================================================

if GEN_DIR not in sys.path:
    sys.path.insert(0, GEN_DIR)

if OWW_DIR not in sys.path:
    sys.path.insert(0, OWW_DIR)

missing_mods = []

modules = [
    'torch',
    'torchinfo',
    'torchmetrics',
    'scipy',
    'numpy',
    'tqdm',
    'yaml',
    'mutagen',
    'pronouncing',
    'torchaudio',
    'audiomentations',
    'torch_audiomentations',
    'speechbrain',
    'acoustics',
    'onnx',
    'onnxruntime',
    'soundfile',
    'requests',
    'huggingface_hub',
    'piper',
    'openwakeword',
]

for mod in modules:

    try:
        importlib.import_module(mod)
    except Exception as e:
        missing_mods.append((mod, str(e)))
        print(
            f'  ✗ IMPORT FAIL: {mod} '
            f'({type(e).__name__}: {e})'
        )

# ============================================================
# Piper compatibility test
# ============================================================

try:

    from piper import PiperVoice, SynthesisConfig
    from piper.phonemize_espeak import EspeakPhonemizer

    print('  ✓ Piper imports OK')

    for name, path in [
        ('Paola', f'{VOICE_DIR}/it_IT-paola-medium.onnx'),
        ('Riccardo', f'{VOICE_DIR}/it_IT-riccardo-x_low.onnx'),
    ]:
        voice = PiperVoice.load(path, use_cuda=False)
        print(
            f'  ✓ {name}: '
            f'{voice.config.sample_rate} Hz, '
            f'{voice.config.num_speakers} speaker'
        )

except Exception as e:

    print(
        f'  ✗ Piper voice load failed: '
        f'{type(e).__name__}: {e}'
    )

    missing_mods.append(('piper Italian voices', str(e)))

# ============================================================
# Generator adapter test
# ============================================================

try:

    from generate_samples import generate_samples

    print('  ✓ Italian generate_samples adapter imports')

except Exception as e:

    print(
        f'  ✗ generator adapter failed: '
        f'{type(e).__name__}: {e}'
    )

    missing_mods.append(('generate_samples adapter', str(e)))

# ============================================================
# OpenWakeWord imports
# ============================================================

try:

    from openwakeword.data import (
        generate_adversarial_texts,
        augment_clips,
        mmap_batch_generator
    )

    from openwakeword.utils import (
        compute_features_from_generator,
        AudioFeatures
    )

    print('  ✓ openWakeWord data/utils import OK')

except Exception as e:

    print(
        f'  ✗ openWakeWord internal import failed: '
        f'{type(e).__name__}: {e}'
    )

    missing_mods.append(('openwakeword.data/utils', str(e)))

# ============================================================
# Verify Italian negative patch
# ============================================================

with open(TRAIN_PY, 'r', encoding='utf-8') as f:
    train_content = f.read()

if '# ITALIAN NEGATIVE PHRASE PATCH' in train_content:
    print('  ✓ English adversarial generator disabled')
else:
    print('  ✗ Italian negative patch not found')
    missing_mods.append(
        ('Italian negative patch', 'marker missing in train.py')
    )

# ============================================================
# GPU
# ============================================================

print(
    f'\n  CUDA: {torch.cuda.is_available()}, '
    f'GPU: '
    f'{torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-"}'
)

# ============================================================
# Final gate
# ============================================================

if missing_files or missing_mods:

    raise RuntimeError(
        f'Pre-flight failed — '
        f'{len(missing_files)} missing files, '
        f'{len(missing_mods)} import/compatibility errors.'
    )

print('\n  PRE-FLIGHT PASSED — Italian Nova training environment is ready.')

  ✓ Italian Paola ONNX: 63.5 MB
  ✓ Italian Paola JSON: 7.1 KB
  ✓ Italian Riccardo ONNX: 28.1 MB
  ✓ Italian Riccardo JSON: 4.2 KB
  ✓ Italian generate_samples adapter: 2.6 KB
  ✓ openwakeword train.py: 45.5 KB
  ✓ openwakeword data.py: 46.6 KB
  ✓ openwakeword utils.py: 33.1 KB
  ✓ Piper imports OK
  ✓ Paola: 22050 Hz, 1 speaker
  ✓ Riccardo: 16000 Hz, 1 speaker
  ✓ Italian generate_samples adapter imports
  ✓ openWakeWord data/utils import OK
  ✓ English adversarial generator disabled

  CUDA: True, GPU: NVIDIA A100-SXM4-80GB

  PRE-FLIGHT PASSED — Italian Nova training environment is ready.


## 5. Download openwakeword shared models (mel + embedding)

Required by augment_clips' feature-extraction step. Idempotent â€” skips files already cached.

In [125]:
import os
models_dir = '/content/openwakeword/openwakeword/resources/models'
os.makedirs(models_dir, exist_ok=True)
BASE = 'https://github.com/dscripka/openWakeWord/releases/download/v0.5.1'
for fname in ['embedding_model.onnx', 'embedding_model.tflite',
              'melspectrogram.onnx', 'melspectrogram.tflite']:
    out = f'{models_dir}/{fname}'
    if os.path.exists(out) and os.path.getsize(out) > 1000:
        print(f'  cached: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')
    else:
        !wget -q --tries=5 --timeout=120 "{BASE}/{fname}" -O "{out}"
        print(f'  âœ“ downloaded: {fname} ({os.path.getsize(out)/1e3:.0f} KB)')

  cached: embedding_model.onnx (1327 KB)
  cached: embedding_model.tflite (1330 KB)
  cached: melspectrogram.onnx (1088 KB)
  cached: melspectrogram.tflite (1093 KB)


## 6. MIT impulse responses â†’ 16-kHz WAVs (~1 min)

Caches the HF dataset, then converts to WAV. Both halves idempotent.

In [126]:
import os, time, numpy as np
import scipy.io.wavfile as wavfile
from huggingface_hub import snapshot_download
import datasets

output_dir = '/content/mit_rirs'
os.makedirs(output_dir, exist_ok=True)
if len([f for f in os.listdir(output_dir) if f.endswith('.wav')]) >= 250:
    print(f'  cached: {len(os.listdir(output_dir))} MIT IR WAVs')
else:
    # Pre-cache via snapshot_download (more reliable than load_dataset's internal
    # 10s timeout on 270 file metadata calls).
    for i in range(6):
        try:
            snapshot_download(repo_id='davidscripka/MIT_environmental_impulse_responses',
                              repo_type='dataset', etag_timeout=120, max_workers=4)
            break
        except Exception as e:
            wait = 15 * (i+1)
            print(f'  snapshot_download attempt {i+1}/6: {type(e).__name__}: {str(e)[:80]} â€” sleep {wait}s')
            time.sleep(wait)
    # Load + convert
    for attempt in range(4):
        try:
            rir_dataset = datasets.load_dataset('davidscripka/MIT_environmental_impulse_responses',
                                                split='train', streaming=False)
            break
        except Exception as e:
            print(f'  load_dataset attempt {attempt+1}: {type(e).__name__}: {str(e)[:120]}')
            time.sleep(20 * (attempt + 1))
    else:
        raise RuntimeError('load_dataset failed for MIT IRs')
    n = 0
    for i, row in enumerate(rir_dataset):
        out = f'{output_dir}/{i:04d}.wav'
        if os.path.exists(out): continue
        audio = row['audio']
        sr = audio['sampling_rate']
        arr = np.asarray(audio['array'])
        if sr != 16000:
            from scipy.signal import resample_poly
            arr = resample_poly(arr, 16000, sr)
        arr = (arr * 32767).clip(-32768, 32767).astype(np.int16)
        wavfile.write(out, 16000, arr); n += 1
    print(f'  âœ“ wrote {n} new WAVs (total {len(os.listdir(output_dir))})')

  cached: 270 MIT IR WAVs


## 7. Download FMA + ACAV features (~10-15 min)

FMA = ~8 GB zip of MP3s for background-noise mixing. ACAV = ~17 GB .npy of pre-computed features (becomes negative training corpus). Both use stream + resume so a session restart picks up where it left off.

In [127]:
import os, requests, zipfile
from tqdm.auto import tqdm
os.chdir('/content')

# â”€â”€ ACAV features (17 GB) â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
ACAV_FULL = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
ACAV_URL = 'https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
if os.path.exists(ACAV_FULL) and os.path.getsize(ACAV_FULL) > 16_000_000_000:
    print(f'  cached: ACAV {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')
else:
    resume_byte = os.path.getsize(ACAV_FULL) if os.path.exists(ACAV_FULL) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  ACAV: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(ACAV_URL, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(ACAV_FULL, mode) as f, tqdm(total=total, initial=resume_byte,
                                              unit='B', unit_scale=True, unit_divisor=1024,
                                              desc='ACAV') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  âœ“ ACAV: {os.path.getsize(ACAV_FULL)/1e9:.1f} GB')

# â”€â”€ FMA small (~8 GB zip) with resume + extract â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
FMA_DIR = '/content/fma'
FMA_ZIP = '/content/fma_small.zip'
os.makedirs(FMA_DIR, exist_ok=True)
if os.path.isdir(FMA_DIR) and len(os.listdir(FMA_DIR)) > 100:
    print(f'  cached: FMA {len(os.listdir(FMA_DIR))} entries')
else:
    fma_url = 'https://os.unil.cloud.switch.ch/fma/fma_small.zip'
    resume_byte = os.path.getsize(FMA_ZIP) if os.path.exists(FMA_ZIP) else 0
    headers = {'Range': f'bytes={resume_byte}-'} if resume_byte else {}
    mode = 'ab' if resume_byte else 'wb'
    print(f'  FMA: {"resuming from" if resume_byte else "fetching"} {resume_byte/1e9:.1f} GB')
    with requests.get(fma_url, stream=True, headers=headers, timeout=120) as r:
        r.raise_for_status()
        total = int(r.headers.get('content-length', 0)) + resume_byte
        with open(FMA_ZIP, mode) as f, tqdm(total=total, initial=resume_byte,
                                            unit='B', unit_scale=True, unit_divisor=1024,
                                            desc='FMA') as pbar:
            for chunk in r.iter_content(chunk_size=4*1024*1024):
                if chunk: f.write(chunk); pbar.update(len(chunk))
    print(f'  extracting FMA...')
    with zipfile.ZipFile(FMA_ZIP) as zf:
        for name in tqdm(zf.namelist(), desc='extract', unit='f'):
            zf.extract(name, FMA_DIR)
    os.remove(FMA_ZIP)
    print(f'  âœ“ FMA extracted: {len(os.listdir(FMA_DIR))} entries')

  cached: ACAV 17.3 GB
  FMA: fetching 0.0 GB


FMA:   0%|          | 0.00/7.15G [00:00<?, ?B/s]

  extracting FMA...


extract:   0%|          | 0/8002 [00:00<?, ?f/s]

  âœ“ FMA extracted: 1 entries


## 8. FMA MP3s â†’ 16-kHz mono WAVs (~5 min)

audiomentations needs WAV. 1500 conversions = plenty of background-noise variety.

In [128]:
import os, glob, subprocess
from tqdm.auto import tqdm
out_dir = '/content/fma_wav'
os.makedirs(out_dir, exist_ok=True)
n_existing = len(os.listdir(out_dir))
if n_existing >= 1500:
    print(f'  cached: {n_existing} FMA WAVs')
else:
    mp3s = glob.glob('/content/fma/**/*.mp3', recursive=True)
    if len(mp3s) < 100:
        raise RuntimeError(f'FMA download incomplete â€” only {len(mp3s)} MP3s found')
    existing = set(os.listdir(out_dir))
    n_target = min(1500, len(mp3s))
    to_convert = [(i, mp3s[i]) for i in range(n_target) if f'{i:05d}.wav' not in existing]
    print(f'  Converting {len(to_convert)} MP3s â†’ 16kHz mono WAVs (skipping {n_target - len(to_convert)} cached)')
    for i, mp3 in tqdm(to_convert):
        out = f'{out_dir}/{i:05d}.wav'
        subprocess.run(['ffmpeg', '-y', '-loglevel', 'error', '-i', mp3,
                        '-ac', '1', '-ar', '16000', out], check=False)
    print(f'  âœ“ {len(os.listdir(out_dir))} FMA WAVs ready')

  Converting 1 MP3s â†’ 16kHz mono WAVs (skipping 1499 cached)


  0%|          | 0/1 [00:00<?, ?it/s]

  âœ“ 1499 FMA WAVs ready


## 9. Subsample ACAV â€” 1.7 GB train + 170 MB val

Full 17 GB would OOM training pass. **Validation slice stays as raw `(M, 96)`** â€” the hand-rolled trainer slides a 16-frame window over it itself.

In [129]:
import numpy as np, os
SRC = '/content/openwakeword_features_ACAV100M_2000_hrs_16bit.npy'
TRAIN_DST = '/content/acav_train_subset.npy'
VAL_DST   = '/content/acav_val_subset.npy'
if os.path.exists(TRAIN_DST) and os.path.exists(VAL_DST):
    train_arr = np.load(TRAIN_DST, mmap_mode='r')
    val_arr = np.load(VAL_DST, mmap_mode='r')
    print(f'  cached: train={train_arr.shape} ({train_arr.nbytes/1e9:.2f} GB), val={val_arr.shape} ({val_arr.nbytes/1e6:.0f} MB)')
else:
    if not os.path.exists(SRC):
        raise RuntimeError(f'ACAV source missing: {SRC} â€” re-run cell 7')
    arr = np.load(SRC, mmap_mode='r')
    print(f'  source: {arr.shape}, dtype={arr.dtype}')
    n_total = len(arr)
    n_train = n_total // 10
    n_val_rows = n_total // 100
    print(f'  slicing: train=[:{n_train}] (3-D), val=[{n_train}:{n_train+n_val_rows}] (flattened to 2-D)')
    train_chunk = np.array(arr[:n_train])
    np.save(TRAIN_DST, train_chunk)
    print(f'  âœ“ train: {train_chunk.shape} ({train_chunk.nbytes/1e9:.2f} GB)')
    del train_chunk
    val_chunk = np.array(arr[n_train:n_train+n_val_rows])
    val_flat = val_chunk.reshape(-1, val_chunk.shape[-1])
    np.save(VAL_DST, val_flat)
    print(f'  âœ“ val: {val_flat.shape} ({val_flat.nbytes/1e6:.0f} MB)')
    del val_chunk, val_flat
    os.remove(SRC)
    print('  removed 17 GB original')

  cached: train=(562500, 16, 96) (1.73 GB), val=(900000, 96) (173 MB)


## 10. Build training config (no example-yaml dependency)

Self-contained â€” every field set explicitly. Doesn't depend on upstream's `examples/` directory which has been renamed/removed across openwakeword versions.

In [141]:
import yaml
import os

# ============================================================
# NOVA ITALIANA
# ============================================================

TARGET_PHRASE = [
    'nova',
    'nova!',
    'nova?'
]

MODEL_NAME = 'nova_it'

# ============================================================
# Italian hard negatives
#
# Sono volutamente parole/frasi foneticamente vicine a "Nova".
# Il generatore avversariale inglese è stato disattivato.
# ============================================================

CUSTOM_NEGATIVES = [

    # Molto vicini
    'nove',
    'nuova',
    'nuovo',
    'Nora',
    'Nola',
    'Noa',
    'Non va',
    'Non va bene',
    'Non va più',
    'Non va mai',

    # Stessa coda /ova/
    'prova',
    'trova',
    'prova ora',
    'trova ora',
    'prova qui',
    'trova qui',
    'prova questo',
    'trova questo',

    # Parole italiane brevi / foneticamente vicine
    'nonna',
    'nota',
    'notte',
    'nodo',
    'nora qui',
    'nola qui',
    'nuova casa',
    'nuova cosa',
    'nuova luce',
    'nuova versione',
    'nuovo comando',
    'nuovo modello',
    'nove volte',

    # Frasi realistiche che possono essere dette in casa
    'non lo so',
    'non lo voglio',
    'non va così',
    'non va bene così',
    'prova a vedere',
    'prova a farlo',
    'trova il telecomando',
    'trova la chiave',
    'prova ancora',
    'prova di nuovo',
    'nuova lampada',
    'nuova canzone',
    'nuova stanza',
]

# ============================================================
# Config
# ============================================================

config = {

    # Wake word
    'target_phrase': TARGET_PHRASE,
    'model_name': MODEL_NAME,
    'custom_negative_phrases': CUSTOM_NEGATIVES,

    # Synthetic clips
    'n_samples': 20000,
    'n_samples_val': 5000,
    'tts_batch_size': 50,

    # Italian Piper adapter
    'piper_sample_generator_path':
        '/content/piper_italian_generator',

    # Augmentation
    'augmentation_rounds': 1,
    'augmentation_batch_size': 16,

    # Training
    'steps': 20000,
    'max_negative_weight': 1500,

    # Informational targets
    'target_accuracy': 0.90,
    'target_recall': 0.85,
    'target_false_positives_per_hour': 0.5,

    'batch_size': 128,
    'learning_rate': 1e-4,

    # Model architecture
    'model_type': 'dnn',
    'layer_dim': 128,
    'layer_size': 128,
    'n_blocks': 1,
    'model_input_shape': [16, 96],
    'n_classes': 1,

    'batch_n_per_class': {
        'ACAV100M_sample': 1024,
        'adversarial_negative': 50,
        'positive': 50,
    },

    # Data paths
    'background_paths': [
        '/content/fma_wav'
    ],

    'background_paths_duplication_rate': [1],

    'rir_paths': [
        '/content/mit_rirs'
    ],

    'false_positive_validation_data_path':
        '/content/acav_val_subset.npy',

    'feature_data_files': {
        'ACAV100M_sample':
            '/content/acav_train_subset.npy'
    },

    # Output
    'output_dir':
        f'/content/{MODEL_NAME}_output',

    'tflite_export': False,
    'onnx_export': True,

    # Clip dirs
    'positive_clips_train_dir':
        f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_train',

    'positive_clips_test_dir':
        f'/content/{MODEL_NAME}_output/{MODEL_NAME}/positive_test',

    'negative_clips_train_dir':
        f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_train',

    'negative_clips_test_dir':
        f'/content/{MODEL_NAME}_output/{MODEL_NAME}/negative_test',

    'feature_save_dir':
        f'/content/{MODEL_NAME}_output/{MODEL_NAME}',
}

os.makedirs(config['output_dir'], exist_ok=True)

with open('/content/my_model.yaml', 'w', encoding='utf-8') as f:
    yaml.dump(
        config,
        f,
        sort_keys=False,
        allow_unicode=True
    )

print(f'  target_phrase: {config["target_phrase"]}')
print(f'  model_name:    {config["model_name"]}')
print(f'  negatives:     {len(config["custom_negative_phrases"])} Italian phrases')
print(f'  steps:         {config["steps"]}')
print(f'  max_neg_w:     {config["max_negative_weight"]}')
print(f'  target_recall: {config["target_recall"]}')
print(f'  target_fp/hr:  {config["target_false_positives_per_hour"]}')

  target_phrase: ['nova', 'nova!', 'nova?']
  model_name:    nova_it
  negatives:     46 Italian phrases
  steps:         20000
  max_neg_w:     1500
  target_recall: 0.85
  target_fp/hr:  0.5


In [142]:
import os
import shutil
import yaml

with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)

# Cartelle dei clip generate dalla pipeline
clip_dirs = [
    cfg['positive_clips_train_dir'],
    cfg['positive_clips_test_dir'],
    cfg['negative_clips_train_dir'],
    cfg['negative_clips_test_dir'],
]

# Feature generate dalla precedente configurazione
feature_files = [
    os.path.join(cfg['feature_save_dir'], 'positive_features_train.npy'),
    os.path.join(cfg['feature_save_dir'], 'negative_features_train.npy'),
    os.path.join(cfg['feature_save_dir'], 'positive_features_test.npy'),
    os.path.join(cfg['feature_save_dir'], 'negative_features_test.npy'),
]

print("=== ELIMINAZIONE VECCHI DATI ===")

for path in clip_dirs:
    if os.path.isdir(path):
        shutil.rmtree(path)
        print("ELIMINATA:", path)
    else:
        print("NON PRESENTE:", path)

for path in feature_files:
    if os.path.isfile(path):
        os.remove(path)
        print("ELIMINATO:", path)
    else:
        print("NON PRESENTE:", path)

print("\nPulizia completata.")
print("n_samples =", cfg['n_samples'])
print("n_samples_val =", cfg['n_samples_val'])

=== ELIMINAZIONE VECCHI DATI ===
ELIMINATA: /content/nova_it_output/nova_it/positive_train
ELIMINATA: /content/nova_it_output/nova_it/positive_test
ELIMINATA: /content/nova_it_output/nova_it/negative_train
ELIMINATA: /content/nova_it_output/nova_it/negative_test
ELIMINATO: /content/nova_it_output/nova_it/positive_features_train.npy
ELIMINATO: /content/nova_it_output/nova_it/negative_features_train.npy
ELIMINATO: /content/nova_it_output/nova_it/positive_features_test.npy
ELIMINATO: /content/nova_it_output/nova_it/negative_features_test.npy

Pulizia completata.
n_samples = 20000
n_samples_val = 5000


## 11. Generate Piper TTS clips (~10-15 min)

Uses openwakeword's --generate_clips runner which reads from my_model.yaml. Idempotent â€” re-running finds existing clips and skips.

In [144]:
import sys
import os
import yaml
import subprocess

with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)

dirs = {
    'positive_train': (cfg['positive_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'positive_test':  (cfg['positive_clips_test_dir'], int(cfg['n_samples_val'] * 0.75)),
    'negative_train': (cfg['negative_clips_train_dir'], int(cfg['n_samples'] * 0.75)),
    'negative_test':  (cfg['negative_clips_test_dir'], int(cfg['n_samples_val'] * 0.75)),
}

status = []

for name, (path, expected) in dirs.items():
    n = len(os.listdir(path)) if os.path.isdir(path) else 0
    ok = n >= expected
    status.append((name, n, expected, ok))
    print(f'  {"PASS" if ok else "MISS"} {name}: {n} clips (expect >={expected})')

all_full = all(ok for _, _, _, ok in status)

if all_full:
    print('\n  cached: all 4 clip dirs already populated')

else:
    print('\n  -> running generate_clips')

    env = os.environ.copy()
    env["PYTHONPATH"] = "/content/openwakeword:" + env.get("PYTHONPATH", "")

    cmd = [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        "/content/my_model.yaml",
        "--generate_clips"
    ]

    result = subprocess.run(
        cmd,
        env=env,
        text=True
    )

    if result.returncode != 0:
        raise RuntimeError(
            f"generate_clips failed with exit code {result.returncode}"
        )

    for name, (path, expected) in dirs.items():
        n = len(os.listdir(path)) if os.path.isdir(path) else 0
        assert n >= expected, (
            f'generate_clips left {name} with only {n} clips '
            f'(expected >={expected})'
        )

    print('\n  PASS all 4 clip dirs now populated')

  MISS positive_train: 0 clips (expect >=15000)
  MISS positive_test: 0 clips (expect >=3750)
  MISS negative_train: 0 clips (expect >=15000)
  MISS negative_test: 0 clips (expect >=3750)

  -> running generate_clips

  PASS all 4 clip dirs now populated


## 12. Resample TTS clips 22050 â†’ 16000 Hz

Piper TTS outputs at libritts's native 22050 Hz. The augment + feature pipelines expect 16 kHz.

In [145]:
import os, glob, soundfile as sf, yaml
from scipy.signal import resample_poly
from tqdm.auto import tqdm
TARGET_SR = 16000
with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)
ROOT = cfg['output_dir']
wav_dirs = sorted({os.path.dirname(f)
                   for f in glob.glob(f'{ROOT}/**/*.wav', recursive=True)})
for d in wav_dirs:
    files = [f for f in os.listdir(d) if f.endswith('.wav')]
    if not files: continue
    sr_counts = {}
    for f in files[:20]:
        sr = sf.info(f'{d}/{f}').samplerate
        sr_counts[sr] = sr_counts.get(sr, 0) + 1
    if all(k == TARGET_SR for k in sr_counts):
        print(f'  ok {d}: {len(files)} files at 16 kHz (skip)')
        continue
    print(f'  resampling {d}: {len(files)} files, SRs {sr_counts}')
    n = 0
    for f in tqdm(files, desc=os.path.basename(d) or d, leave=False):
        p = f'{d}/{f}'
        data, sr = sf.read(p)
        if sr == TARGET_SR: continue
        new_data = resample_poly(data.astype('float32'), TARGET_SR, sr)
        sf.write(p, new_data, TARGET_SR); n += 1
    print(f'  ok {d}: resampled {n}/{len(files)}')
# Clear stale .npy features so augment re-runs cleanly
for f in glob.glob(f'{ROOT}/**/*.npy', recursive=True):
    os.remove(f)
print('\n  cleared stale features (will be regenerated by augment)')

  resampling /content/nova_it_output/nova_it/negative_test: 5000 files, SRs {22050: 12, 16000: 8}


negative_test:   0%|          | 0/5000 [00:00<?, ?it/s]

  ok /content/nova_it_output/nova_it/negative_test: resampled 2501/5000
  resampling /content/nova_it_output/nova_it/negative_train: 20000 files, SRs {22050: 10, 16000: 10}


negative_train:   0%|          | 0/20000 [00:00<?, ?it/s]

  ok /content/nova_it_output/nova_it/negative_train: resampled 10001/20000
  resampling /content/nova_it_output/nova_it/positive_test: 5000 files, SRs {22050: 12, 16000: 8}


positive_test:   0%|          | 0/5000 [00:00<?, ?it/s]

  ok /content/nova_it_output/nova_it/positive_test: resampled 2501/5000
  resampling /content/nova_it_output/nova_it/positive_train: 20000 files, SRs {22050: 9, 16000: 11}


positive_train:   0%|          | 0/20000 [00:00<?, ?it/s]

  ok /content/nova_it_output/nova_it/positive_train: resampled 10001/20000

  cleared stale features (will be regenerated by augment)


## 13. Augment + featurise (~10 min)

Outputs `(N, 16, 96)` feature .npy files that the hand-rolled trainer reads.

In [146]:
import subprocess
import os
import sys

env = os.environ.copy()
env["PYTHONPATH"] = "/content/openwakeword:" + env.get("PYTHONPATH", "")

result = subprocess.run(
    [
        sys.executable,
        "/content/openwakeword/openwakeword/train.py",
        "--training_config",
        "/content/my_model.yaml",
        "--augment_clips"
    ],
    env=env,
    text=True
)

if result.returncode != 0:
    raise RuntimeError(
        f"augment_clips failed with exit code {result.returncode}"
    )

## 14. Hand-rolled trainer (~30-40 min on L4 / A100)

Mirrors openWakeWord's `auto_train`: 3-stage lr schedule (1e-4 â†’ 1e-5 â†’ 1e-6), neg_weight ramp 1â†’1500, hard-negative mining, FP/hour validation against ACAV val, percentile-based checkpoint saves.

In [147]:
import os, copy, math, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import yaml

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

with open('/content/my_model.yaml') as f:
    cfg = yaml.safe_load(f)

FEAT = cfg['feature_save_dir']

pos_train = torch.from_numpy(
    np.load(f'{FEAT}/positive_features_train.npy').astype(np.float32)
).to(DEVICE)

neg_train = torch.from_numpy(
    np.load(f'{FEAT}/negative_features_train.npy').astype(np.float32)
).to(DEVICE)

pos_test = torch.from_numpy(
    np.load(f'{FEAT}/positive_features_test.npy').astype(np.float32)
).to(DEVICE)

neg_test = torch.from_numpy(
    np.load(f'{FEAT}/negative_features_test.npy').astype(np.float32)
).to(DEVICE)

print(
    f'  pos_train={tuple(pos_train.shape)}  '
    f'neg_train={tuple(neg_train.shape)}'
)

print(
    f'  pos_test={tuple(pos_test.shape)}  '
    f'neg_test={tuple(neg_test.shape)}'
)

# ============================================================
# ACAV training / validation
# ============================================================

acav_train_np = np.load(
    cfg['feature_data_files']['ACAV100M_sample'],
    mmap_mode='r'
)

print(f'  acav_train (mmap): {acav_train_np.shape}')

acav_val_np = np.load(
    cfg['false_positive_validation_data_path']
)

M = acav_val_np.shape[0]

val_listen_hours = M * 0.08 / 3600.0

print(
    f'  acav_val: {acav_val_np.shape} '
    f'→ {val_listen_hours:.2f} hr listening time'
)

n_win_val = M - 16

acav_val_windows = np.lib.stride_tricks.sliding_window_view(
    acav_val_np,
    (16, 96)
)[:, 0, :, :]

acav_val_windows = np.ascontiguousarray(
    acav_val_windows.astype(np.float32)
)

print(
    f'  acav_val_windows: {acav_val_windows.shape} '
    f'({acav_val_windows.nbytes / 1e6:.0f} MB)'
)

VAL_BATCH = 4096


# ============================================================
# Model
# ============================================================

class WakewordModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.flatten = nn.Flatten()

        self.layer1 = nn.Linear(
            16 * 96,
            128
        )

        self.layernorm1 = nn.LayerNorm(128)

        self.relu1 = nn.ReLU()

        self.layer2 = nn.Linear(
            128,
            1
        )

    def forward(self, x):

        return self.layer2(
            self.relu1(
                self.layernorm1(
                    self.layer1(
                        self.flatten(x)
                    )
                )
            )
        )


model = WakewordModel().to(DEVICE)

loss_fn = nn.BCEWithLogitsLoss(
    reduction='none'
)


# ============================================================
# Training parameters
# ============================================================

TOTAL_STEPS = cfg['steps']
MAX_NEG_W = cfg['max_negative_weight']
TARGET_FP_PER_HR = cfg['target_false_positives_per_hour']

THRESH = 0.5

history = {
    'val_recall': [],
    'val_accuracy': [],
    'val_fp_per_hour': [],
    'val_n_fp': [],
    'loss': []
}

best_models = []


# ============================================================
# Batch sizes
# ============================================================

B_POS = 32
B_ANEG = 32
B_ACAV = 64


# ============================================================
# ACAV random window sampler
# ============================================================

def random_acav_window_batch(k):

    N, T, F = acav_train_np.shape

    rows = np.random.randint(
        0,
        N,
        size=k
    )

    starts = np.random.randint(
        0,
        T - 16 + 1,
        size=k
    )

    out = np.empty(
        (k, 16, F),
        dtype=np.float32
    )

    for i, (r, s) in enumerate(
        zip(rows, starts)
    ):

        out[i] = acav_train_np[
            r,
            s:s + 16,
            :
        ].astype(np.float32)

    return out


# ============================================================
# Build training batch
# ============================================================

def build_batch():

    p_idx = torch.randint(
        0,
        pos_train.shape[0],
        (B_POS,),
        device=DEVICE
    )

    aneg_idx = torch.randint(
        0,
        neg_train.shape[0],
        (B_ANEG,),
        device=DEVICE
    )

    p = pos_train[p_idx]
    an = neg_train[aneg_idx]

    acav = torch.from_numpy(
        random_acav_window_batch(B_ACAV)
    ).to(DEVICE)

    x = torch.cat(
        [
            p,
            an,
            acav
        ],
        dim=0
    )

    y = torch.cat(
        [
            torch.ones(
                B_POS,
                device=DEVICE
            ),
            torch.zeros(
                B_ANEG + B_ACAV,
                device=DEVICE
            )
        ]
    )

    return x, y


# ============================================================
# Validation
# ============================================================

@torch.no_grad()
def validate(step_label):

    model.eval()

    p_preds = torch.sigmoid(
        model(pos_test)
    ).squeeze(-1)

    n_preds = torch.sigmoid(
        model(neg_test)
    ).squeeze(-1)

    recall = (
        (p_preds >= THRESH)
        .float()
        .mean()
        .item()
    )

    accuracy = (
        (
            (p_preds >= THRESH).sum()
            +
            (n_preds < THRESH).sum()
        ).item()
        /
        (
            pos_test.shape[0]
            +
            neg_test.shape[0]
        )
    )

    n_fp = 0

    for i in range(
        0,
        n_win_val,
        VAL_BATCH
    ):

        chunk = torch.from_numpy(
            acav_val_windows[
                i:i + VAL_BATCH
            ]
        ).to(DEVICE)

        n_fp += (
            torch.sigmoid(
                model(chunk)
            )
            .squeeze(-1)
            >= THRESH
        ).sum().item()

    fp_per_hour = (
        n_fp /
        max(
            val_listen_hours,
            1e-6
        )
    )

    history['val_recall'].append(
        recall
    )

    history['val_accuracy'].append(
        accuracy
    )

    history['val_fp_per_hour'].append(
        fp_per_hour
    )

    history['val_n_fp'].append(
        n_fp
    )

    save = False

    if len(history['val_n_fp']) >= 3:

        fp_p50 = np.percentile(
            history['val_n_fp'],
            50
        )

        rc_p5 = np.percentile(
            history['val_recall'],
            5
        )

        if (
            n_fp <= fp_p50
            and
            recall >= rc_p5
        ):

            best_models.append(
                (
                    copy.deepcopy(
                        model.state_dict()
                    ),
                    {
                        'val_recall': recall,
                        'val_accuracy': accuracy,
                        'val_fp_per_hour': fp_per_hour,
                        'val_n_fp': n_fp
                    }
                )
            )

            save = True

    print(
        f'  [{step_label}] '
        f'recall={recall:.3f}  '
        f'acc={accuracy:.3f}  '
        f'fp/hr={fp_per_hour:.2f}  '
        f'saved={"+" if save else "-"}',
        flush=True
    )

    model.train()


# ============================================================
# Training stage
# ============================================================

def run_stage(
    stage_idx,
    n_steps,
    lr,
    max_neg_w,
    val_window_frac=1.0
):

    optimizer = optim.Adam(
        model.parameters(),
        lr=lr
    )

    weight_schedule = np.linspace(
        1.0,
        max_neg_w,
        n_steps
    )

    val_start = int(
        n_steps *
        (1.0 - val_window_frac)
    )

    val_steps = set(
        np.linspace(
            val_start,
            n_steps - 1,
            20
        ).astype(int)
    )

    warmup = max(
        1,
        n_steps // 5
    )

    hold = n_steps // 3

    accumulated = []

    print(
        f'\n=== Stage {stage_idx}: '
        f'{n_steps} steps, '
        f'lr={lr}, '
        f'max_neg_w={max_neg_w} ===',
        flush=True
    )

    t0 = time.time()

    for step in range(n_steps):

        # ----------------------------------------------------
        # Learning-rate schedule
        # ----------------------------------------------------

        if step < warmup:

            lr_now = (
                lr *
                (step + 1) /
                warmup
            )

        elif step < warmup + hold:

            lr_now = lr

        else:

            decay_t = (
                step -
                warmup -
                hold
            ) / max(
                1,
                n_steps -
                warmup -
                hold
            )

            lr_now = (
                lr *
                0.5 *
                (
                    1.0 +
                    math.cos(
                        math.pi *
                        min(
                            1.0,
                            decay_t
                        )
                    )
                )
            )

        for pg in optimizer.param_groups:

            pg['lr'] = lr_now


        # ----------------------------------------------------
        # Build batch
        # ----------------------------------------------------

        x, y = build_batch()

        logits = model(x).squeeze(-1)

        preds = torch.sigmoid(logits)

        keep = (
            (
                (y == 0)
                &
                (preds >= 0.001)
            )
            |
            (
                (y == 1)
                &
                (preds < 0.999)
            )
        )

        if keep.sum() == 0:

            if step in val_steps:

                validate(
                    f'stage{stage_idx} '
                    f'step {step}/{n_steps}'
                )

            continue


        # ----------------------------------------------------
        # Accumulate hard examples
        # ----------------------------------------------------

        kept_logits = logits[keep]

        kept_y = y[keep]

        neg_w = weight_schedule[step]

        w = torch.where(
            kept_y > 0.5,
            torch.tensor(
                1.0,
                device=DEVICE
            ),
            torch.tensor(
                neg_w,
                device=DEVICE,
                dtype=torch.float32
            )
        )

        accumulated.append(
            (
                kept_logits,
                kept_y,
                w
            )
        )


        # ----------------------------------------------------
        # Optimizer update
        # ----------------------------------------------------

        if sum(
            t[0].shape[0]
            for t in accumulated
        ) >= 128:

            cat_logits = torch.cat(
                [
                    t[0]
                    for t in accumulated
                ]
            )

            cat_y = torch.cat(
                [
                    t[1]
                    for t in accumulated
                ]
            )

            cat_w = torch.cat(
                [
                    t[2]
                    for t in accumulated
                ]
            )

            loss = (
                loss_fn(
                    cat_logits,
                    cat_y
                )
                *
                cat_w
            ).mean()

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            history['loss'].append(
                loss.item()
            )

            accumulated.clear()


        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        if step in val_steps:

            elapsed = (
                time.time() -
                t0
            ) / 60

            validate(
                f'stage{stage_idx} '
                f'step {step}/{n_steps} '
                f'({elapsed:.1f}m, '
                f'lr={lr_now:.6f}, '
                f'neg_w={neg_w:.0f})'
            )


    print(
        f'  Stage {stage_idx} done in '
        f'{(time.time() - t0) / 60:.1f} min',
        flush=True
    )


# ============================================================
# IMPORTANT:
# 30,000 TOTAL TRAINING STEPS
#
# Stage 1 = 10,000
# Stage 2 = 10,000
# Stage 3 = 10,000
# ============================================================

stage1_steps = 10000
stage2_steps = 10000
stage3_steps = 10000

max_neg_w_now = MAX_NEG_W


# ============================================================
# Stage 1
# ============================================================

run_stage(
    1,
    stage1_steps,
    lr=1e-4,
    max_neg_w=max_neg_w_now,
    val_window_frac=0.25
)


# ============================================================
# Adaptive negative weighting
# ============================================================

if (
    history['val_fp_per_hour']
    and
    min(
        history['val_fp_per_hour']
    ) > TARGET_FP_PER_HR
):

    max_neg_w_now *= 2

    print(
        f'\n[adapt] stage1 best FP/hr '
        f'> target → max_neg_w doubled '
        f'to {max_neg_w_now}'
    )


# ============================================================
# Stage 2
# ============================================================

run_stage(
    2,
    stage2_steps,
    lr=1e-5,
    max_neg_w=max_neg_w_now,
    val_window_frac=1.0
)


# ============================================================
# Adaptive negative weighting
# ============================================================

if (
    history['val_fp_per_hour']
    and
    min(
        history['val_fp_per_hour']
    ) > TARGET_FP_PER_HR
):

    max_neg_w_now *= 2

    print(
        f'\n[adapt] stage2 best FP/hr '
        f'> target → max_neg_w doubled '
        f'to {max_neg_w_now}'
    )


# ============================================================
# Stage 3
# ============================================================

run_stage(
    3,
    stage3_steps,
    lr=1e-6,
    max_neg_w=max_neg_w_now,
    val_window_frac=1.0
)


# ============================================================
# Final results
# ============================================================

print(
    f'\n=== Training done. '
    f'{len(best_models)} checkpoints saved. ==='
)

print(
    f'  best val_fp_per_hour: '
    f'{min(history["val_fp_per_hour"]):.2f}'
)

print(
    f'  best val_recall:      '
    f'{max(history["val_recall"]):.3f}'
)

print(
    f'  best val_accuracy:    '
    f'{max(history["val_accuracy"]):.3f}'
)

Device: cuda
  pos_train=(20000, 16, 96)  neg_train=(20000, 16, 96)
  pos_test=(5000, 16, 96)  neg_test=(5000, 16, 96)
  acav_train (mmap): (562500, 16, 96)
  acav_val: (900000, 96) → 20.00 hr listening time
  acav_val_windows: (899985, 16, 96) (5530 MB)

=== Stage 1: 10000 steps, lr=0.0001, max_neg_w=1500 ===
  [stage1 step 7500/10000 (0.4m, lr=0.000056, neg_w=1125)] recall=0.084  acc=0.542  fp/hr=0.05  saved=-
  [stage1 step 7631/10000 (0.4m, lr=0.000051, neg_w=1145)] recall=0.096  acc=0.548  fp/hr=0.10  saved=-
  [stage1 step 7763/10000 (0.4m, lr=0.000047, neg_w=1165)] recall=0.092  acc=0.546  fp/hr=0.10  saved=+
  [stage1 step 7894/10000 (0.4m, lr=0.000042, neg_w=1184)] recall=0.118  acc=0.559  fp/hr=0.25  saved=-
  [stage1 step 8026/10000 (0.5m, lr=0.000038, neg_w=1204)] recall=0.087  acc=0.543  fp/hr=0.05  saved=+
  [stage1 step 8157/10000 (0.5m, lr=0.000034, neg_w=1224)] recall=0.099  acc=0.549  fp/hr=0.15  saved=-
  [stage1 step 8289/10000 (0.5m, lr=0.000030, neg_w=1244)] recal

## 15. Ensemble best checkpoints + ONNX export + download

90/90/10 percentile filter on saved checkpoints, average state_dicts, export ONNX with sigmoid baked in, sanity-check, browser-download.

In [139]:
import copy, os
import numpy as np
import torch
if not best_models:
    print('  no saved checkpoints â€” using current model state')
    final_state = {k: v.clone() for k, v in model.state_dict().items()}
else:
    accs = [s['val_accuracy'] for _, s in best_models]
    rcs  = [s['val_recall']   for _, s in best_models]
    fps  = [s['val_fp_per_hour'] for _, s in best_models]
    acc_p90 = np.percentile(accs, 90)
    rc_p90  = np.percentile(rcs, 90)
    fp_p10  = np.percentile(fps, 10)
    print(f'  thresholds: accâ‰¥{acc_p90:.3f}  recallâ‰¥{rc_p90:.3f}  fp/hrâ‰¤{fp_p10:.2f}')
    qualified = [(sd, sc) for sd, sc in best_models
                 if sc['val_accuracy'] >= acc_p90 and sc['val_recall'] >= rc_p90
                 and sc['val_fp_per_hour'] <= fp_p10]
    print(f'  qualified: {len(qualified)} of {len(best_models)} checkpoints')
    if not qualified:
        sorted_models = sorted(best_models, key=lambda t: (t[1]['val_fp_per_hour'], -t[1]['val_recall']))
        qualified = [sorted_models[0]]
        print('  no checkpoint qualified â€” fell back to single best')
    keys = qualified[0][0].keys()
    final_state = {k: torch.stack([sd[k].float() for sd, _ in qualified]).mean(dim=0) for k in keys}
    qa = np.mean([s['val_accuracy'] for _, s in qualified])
    qr = np.mean([s['val_recall'] for _, s in qualified])
    qf = np.mean([s['val_fp_per_hour'] for _, s in qualified])
    print(f'  ensemble averaged: mean acc={qa:.3f}, recall={qr:.3f}, fp/hr={qf:.2f}')
model.load_state_dict(final_state)
model.eval()

# Export with sigmoid baked in so APK runtime sees [0, 1]
out_path = f"{cfg['feature_save_dir']}/{cfg['model_name']}.onnx"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
class WakewordExportable(torch.nn.Module):
    def __init__(self, base): super().__init__(); self.base = base
    def forward(self, x): return torch.sigmoid(self.base(x))
export_model = WakewordExportable(model).to(DEVICE).eval()
dummy = torch.randn(1, 16, 96, device=DEVICE)
torch.onnx.export(export_model, dummy, out_path,
                  input_names=['onnx::Flatten_0'], output_names=['output'],
                  dynamic_axes={'onnx::Flatten_0': {0: 'batch'}, 'output': {0: 'batch'}},
                  opset_version=14, dynamo=False)
print(f'  âœ“ wrote {out_path} ({os.path.getsize(out_path)/1e3:.0f} KB)')

import onnxruntime as ort
sess = ort.InferenceSession(out_path)
print(f'  input:  {sess.get_inputs()[0].name} {sess.get_inputs()[0].shape}')
print(f'  output: {sess.get_outputs()[0].name} {sess.get_outputs()[0].shape}')
with torch.no_grad():
    p_test_np = pos_test.cpu().numpy()
    p_scores = sess.run(None, {sess.get_inputs()[0].name: p_test_np})[0].flatten()
    print(f'  positive test set: mean={p_scores.mean():.3f}, recall@0.5={(p_scores>=0.5).mean():.3f}')
print(f'  ACAV val FP/hour at 0.5: {history["val_fp_per_hour"][-1]:.2f}')

from google.colab import files
files.download(out_path)
print(f'\nDONE. Wake word "{cfg["model_name"]}" trained.')
print('Drop the downloaded .onnx into your APK / on-device app at:')
print(f'  android/app/src/main/assets/wakewords/{cfg["model_name"]}.onnx  (or wherever your APK looks)')

  thresholds: accâ‰¥0.661  recallâ‰¥0.374  fp/hrâ‰¤0.95
  qualified: 0 of 34 checkpoints
  no checkpoint qualified â€” fell back to single best
  ensemble averaged: mean acc=0.653, recall=0.349, fp/hr=0.60
  âœ“ wrote /content/nova_it_output/nova_it/nova_it.onnx (791 KB)
  input:  onnx::Flatten_0 ['batch', 16, 96]
  output: output ['batch', 1]
  positive test set: mean=0.343, recall@0.5=0.349
  ACAV val FP/hour at 0.5: 1.20


/tmp/ipykernel_1004/556737474.py:40: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(export_model, dummy, out_path,


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


DONE. Wake word "nova_it" trained.
Drop the downloaded .onnx into your APK / on-device app at:
  android/app/src/main/assets/wakewords/nova_it.onnx  (or wherever your APK looks)
